# Expanding (running) statistics

An *expanding* statistic grows its window from the first sample to the current
one: every new observation is included and nothing is ever dropped. This
contrasts with a *rolling* window, which keeps only the most recent *n*
samples and slides forward.

Expanding statistics are the natural choice whenever you want a session-to-date
or all-time metric that keeps improving as more data arrives. screamer provides
a family of them: `ExpandingMean`, `ExpandingStd`, `ExpandingVar`,
`ExpandingSkew`, `ExpandingKurt`, `ExpandingSlope`, `ExpandingSum`,
`ExpandingMax`, `ExpandingMin`, and `ExpandingProd`.

Each is a functor: call it with an array to get the full trajectory, or feed
it one value at a time for a live stream. Call `.reset()` to restart from
scratch, useful when a new session or episode begins.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import ExpandingMean, ExpandingStd

SEED = 17
rng = np.random.default_rng(SEED)

# 250-step random walk: cumulative sum of standard-normal increments.
x = np.cumsum(rng.normal(size=250))

In [ ]:
# Compute the expanding mean and expanding standard deviation.
emean = ExpandingMean()(x)
estd  = ExpandingStd()(x)

fig, ax = plt.subplots(figsize=(9, 3.5))

ax.plot(x, lw=0.8, color="0.6", label="random walk")
ax.plot(emean, lw=1.8, color="steelblue", label="expanding mean")

# Shaded +/- 2 standard-deviation band.  The first sample yields NaN for std
# (a single point has no spread), so the band starts at sample 1;
# matplotlib leaves the gap automatically.
ax.fill_between(
    np.arange(len(x)),
    emean - 2 * estd,
    emean + 2 * estd,
    alpha=0.20,
    color="steelblue",
    label="mean +/- 2 std",
)

ax.set_xlabel("sample")
ax.set_title("Expanding mean and +/- 2 sigma band (all-history estimate)")
ax.legend(fontsize=9)
plt.tight_layout()

## Restarting with `.reset()`

Calling `.reset()` clears all accumulated state so the functor behaves as if
it had never seen any data. The most natural use is resetting at the start of
a new session, episode, or trading day so that session-to-date statistics
accumulate independently for each segment.

Below the series is split in half. The expanding mean is computed on the
first half, then the functor is reset and run on the second half. Notice
that the second segment starts from a single point rather than continuing the
long-run average from the first segment.

In [ ]:
# Split the series into two equal halves and run the expanding mean on each.
mid = len(x) // 2
seg_a, seg_b = x[:mid], x[mid:]

em = ExpandingMean()
mean_a = em(seg_a)   # accumulates over segment A
em.reset()           # clear all state
mean_b = em(seg_b)   # starts fresh from segment B

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 5), sharex=False)

for ax, seg, mean, label in [
    (ax_top, seg_a, mean_a, "Segment A"),
    (ax_bot, seg_b, mean_b, "Segment B  (fresh start after reset)"),
]:
    idx = np.arange(len(seg))
    ax.plot(idx, seg,  lw=0.8, color="0.6", label="signal")
    ax.plot(idx, mean, lw=1.8, color="steelblue", label="expanding mean")
    ax.set_title(label)
    ax.legend(fontsize=9)
    ax.set_xlabel("within-segment sample")

plt.tight_layout()

## Streaming one value at a time

The same functor works in a live stream. Passing a single-element array
returns the updated statistic after each new observation, so it can be
embedded in an event loop or a `Dag` node without any interface change.
Batch and streaming results are guaranteed to be identical.

In [ ]:
# Stream the first five values one at a time and show the running mean.
# Calling with a single-element array returns a scalar directly.
em_live = ExpandingMean()
for i, v in enumerate(x[:5]):
    running = em_live(np.array([v]))
    print(f"sample {i}: x={v:+.4f}  expanding_mean={running:+.4f}")

# The batch result for the same five values must match exactly.
batch_means = ExpandingMean()(x[:5])
print("\nbatch result:", np.round(batch_means, 4))